In [28]:
# scikit-learn is a popular Python library for machine learning.
# Colab already has it installed, so we just "import" the two tools we need.

# TfidfVectorizer turns sentences into numbers a computer can compare.
from sklearn.feature_extraction.text import TfidfVectorizer

# cosine_similarity measures how close two sentences are (0 = nothing alike, 1 = identical).
from sklearn.metrics.pairwise import cosine_similarity

In [29]:
# This is our "knowledge base": the only information the assistant is allowed to use.
# SteadyWheels Driving School is a FICTIONAL business created for this demo.
# Each FAQ has:
#   "question" - the main question
#   "examples" - other ways customers might ask the same thing (improves matching)
#   "answer"   - the reply we give

faqs = [
    {"question": "How much do driving lessons cost?",
     "examples": ["price of lessons", "what are your rates", "how much is a lesson", "lesson fees"],
     "answer": "A single 1-hour lesson costs R350. A package of 10 lessons costs R3,200."},

    {"question": "How long is one driving lesson?",
     "examples": ["lesson duration", "how many minutes is a lesson"],
     "answer": "Each lesson is 1 hour long."},

    {"question": "Do I need a learner's licence before I can book lessons?",
     "examples": ["do I need my learners", "can I drive without a learners licence"],
     "answer": "Yes. You need a valid learner's licence before you can start practical driving lessons."},

    {"question": "Can you help me get my learner's licence?",
     "examples": ["learners test preparation", "help me study for my learners"],
     "answer": "Yes. We offer learner's licence preparation classes and practice tests."},

    {"question": "Do you teach in automatic or manual cars?",
     "examples": ["do you have automatic cars", "manual gearbox lessons"],
     "answer": "We offer lessons in both manual and automatic cars."},

    {"question": "Which areas do you pick students up from?",
     "examples": ["do you pick up in my area", "where do you collect students"],
     "answer": "We pick up students in Kuilsriver, Brackenfell, Bellville and surrounding areas."},

    {"question": "How do I book a driving lesson?",
     "examples": ["make a booking", "schedule a lesson", "how can I book"],
     "answer": "You can book by WhatsApp or by filling in the booking form on our website."},

    {"question": "Can I use your car for my driving test?",
     "examples": ["rent the car for the test", "car hire for driving test"],
     "answer": "Yes. You can rent our car for your driving test, including a warm-up lesson beforehand."},

    {"question": "What happens if I need to cancel a lesson?",
     "examples": ["cancel my booking", "reschedule a lesson", "cancellation policy"],
     "answer": "Please cancel at least 24 hours before your lesson, otherwise the lesson may be charged."},

    {"question": "How many lessons will I need before my test?",
     "examples": ["how many lessons do beginners need"],
     "answer": "It depends on your experience, but most beginners need between 10 and 20 lessons."},

    {"question": "Do you offer lessons on weekends?",
     "examples": ["do you work on saturdays", "weekend lessons", "what are your hours"],
     "answer": "Yes. We offer lessons Monday to Friday from 7:00 to 18:00, and Saturdays from 8:00 to 14:00."},

    {"question": "What payment methods do you accept?",
     "examples": ["can I pay with card", "do you take cash", "EFT payment"],
     "answer": "We accept EFT, card and cash payments."},
]

print(f"Loaded {len(faqs)} FAQs")

Loaded 12 FAQs


In [30]:
# We put every question AND every example phrase into one list to search through.
# "owner" remembers which FAQ each phrase belongs to, so we can find the right answer later.
search_texts = []
owner = []

for index, faq in enumerate(faqs):
    for text in [faq["question"]] + faq["examples"]:
        search_texts.append(text)
        owner.append(index)

# TF-IDF gives each important word a score.
# stop_words="english" ignores common words like "the", "is" and "do",
# so matching focuses on meaningful words like "price", "cancel" or "weekend".
vectorizer = TfidfVectorizer(stop_words="english")

# "fit" learns the vocabulary from our FAQs, "transform" turns them into numbers.
faq_vectors = vectorizer.fit_transform(search_texts)

print(f"Search index built from {len(search_texts)} phrases")

Search index built from 41 phrases


In [31]:
# If the best match scores below this, the assistant admits it doesn't know.
# This is a key "responsible AI" choice: it never guesses or makes up an answer.
THRESHOLD = 0.3

FALLBACK = "I'm not sure about that one. Please contact SteadyWheels Driving School directly and we'll gladly help."

def ask(user_question):
    # Turn the customer's question into numbers, the same way as the FAQs.
    user_vector = vectorizer.transform([user_question])

    # Compare it with every FAQ phrase. We get one similarity score per phrase.
    scores = cosine_similarity(user_vector, faq_vectors)[0]

    # Find the highest score and which phrase it belongs to.
    best = scores.argmax()
    best_score = scores[best]

    # Too weak a match? Hand over to a human instead of guessing.
    if best_score < THRESHOLD:
        return FALLBACK, best_score

    # Otherwise return the answer of the FAQ that phrase belongs to.
    return faqs[owner[best]]["answer"], best_score

In [32]:
questions_to_try = [
    "How much are lessons?",
    "Do you work on Saturdays?",
    "Can I pay with card?",
    "Do I need a learners?",
    "Who won the rugby?",   # off-topic: should get the "not sure" answer
]

for q in questions_to_try:
    answer, score = ask(q)
    print(f"Q: {q}")
    print(f"A: {answer}   (match score: {score:.2f})")
    print()

Q: How much are lessons?
A: A single 1-hour lesson costs R350. A package of 10 lessons costs R3,200.   (match score: 0.53)

Q: Do you work on Saturdays?
A: Yes. We offer lessons Monday to Friday from 7:00 to 18:00, and Saturdays from 8:00 to 14:00.   (match score: 1.00)

Q: Can I pay with card?
A: We accept EFT, card and cash payments.   (match score: 1.00)

Q: Do I need a learners?
A: Yes. You need a valid learner's licence before you can start practical driving lessons.   (match score: 1.00)

Q: Who won the rugby?
A: I'm not sure about that one. Please contact SteadyWheels Driving School directly and we'll gladly help.   (match score: 0.00)



In [33]:
# EVALUATION: we test the assistant on questions where we already know the right answer.
# Each test is (customer question, number of the correct FAQ).
# FAQ numbers start at 0, in the order they appear in Cell 2.
# None means the assistant SHOULD say "I'm not sure" (off-topic questions).
import pandas as pd   # pandas shows results as a neat table

test_cases = [
    ("How much are lessons?", 0),
    ("whats the price", 0),
    ("how long is a lesson", 1),
    ("Do I need a learners?", 2),
    ("can you help me with my learners test", 3),
    ("do you have automatic cars", 4),
    ("Do you pick up in Bellville?", 5),
    ("i want to book", 6),
    ("Can I rent the car for the test?", 7),
    ("Can I cancel my booking?", 8),
    ("how many lessons do I need", 9),
    ("Do you work on Saturdays?", 10),
    ("Can I pay with card?", 11),
    ("How long does it take to drive to Cape Town from Kuilsriver?", None),  # should be "not sure"
    ("Do you teach on Sundays?", None),      # should be "not sure"
    ("Who won the rugby?", None),            # off-topic
    ("What is the weather today?", None),    # off-topic
    ("recommend a good pizza place", None),  # off-topic
]

results = []
for question, expected_faq in test_cases:
    answer, score = ask(question)

    # What SHOULD the answer have been?
    expected_answer = FALLBACK if expected_faq is None else faqs[expected_faq]["answer"]

    results.append({
        "question": question,
        "match score": round(float(score), 2),
        "correct": answer == expected_answer,
    })

report = pd.DataFrame(results)
display(report)

accuracy = report["correct"].mean() * 100
print(f"Accuracy: {accuracy:.0f}% ({report['correct'].sum()} of {len(report)} correct)")

,question,match score,correct
0,How much are lessons?,0.53,True
1,whats the price,0.85,True
2,how long is a lesson,0.85,True
3,Do I need a learners?,1.00,True
4,can you help me with my learners test,0.65,True
5,do you have automatic cars,1.00,True
6,Do you pick up in Bellville?,0.67,True
7,i want to book,1.00,True
8,Can I rent the car for the test?,1.00,True
9,Can I cancel my booking?,1.00,True


Accuracy: 89% (16 of 18 correct)


In [34]:
# A simple chat box: type a question, get an answer. Type "quit" to stop.
print("SteadyWheels FAQ Assistant. Type 'quit' to exit.\n")

while True:
    question = input("You: ")
    if question.strip().lower() == "quit":
        print("Assistant: Goodbye!")
        break
    answer, score = ask(question)
    print(f"Assistant: {answer}\n")

SteadyWheels FAQ Assistant. Type 'quit' to exit.

You: how long does a lesson take?
Assistant: Each lesson is 1 hour long.

You: how much does it cost?
Assistant: A single 1-hour lesson costs R350. A package of 10 lessons costs R3,200.

You: can i buy pizza from you?
Assistant: I'm not sure about that one. Please contact SteadyWheels Driving School directly and we'll gladly help.

You: which payment methods do you use?
Assistant: We accept EFT, card and cash payments.

You: quit
Assistant: Goodbye!
